# 01 — Extract (Task 1)
**المصدر:** بوابة اعتماد للمناقصات الحكومية — https://tenders.etimad.sa
**طريقة الوصول:** Playwright (متصفح headless) — لا يوجد API رسمي مفتوح، والموقع محمي بـ Akamai bot detection
**حالة الترخيص:** لسه تحت المراجعة عبر apiportal.etimad.sa — لو انفتح لنا API رسمي لاحقًا، نرجع نغيّر طريقة الاستخراج

هذا الـ notebook يسوي بس: يفتح الصفحات، يستخرج بطاقات المناقصات، ويحفظها **خام بدون تعديل** في `data/raw/`.
التنظيف والترجمة يصير بـ `02_profile_clean.ipynb` — هنا بس استخراج صحيح ودقيق.


### 1. المكتبات

In [ ]:
from playwright.sync_api import sync_playwright
import re
import json
import time
from datetime import date
from pathlib import Path


### 2. إعدادات ثابتة

- `BASE_URL`: رابط صفحة قوائم المناقصات — **بدون فلترة نشاط/فئة**، عشان نغطي أكبر عدد ممكن من الجهات والقطاعات (الهدف مب بس 10 مصادر — الهدف كل التنوّع الممكن من اعتماد)
- `PAGES_TO_SCRAPE`: عدد الصفحات اللي نسحبها بكل تشغيل — الموقع فيه أكثر من 1265 صفحة إجمالي، فنحدد عدد معقول بكل مرة ونزيده تدريجيًا
- `RAW_DIR`: وين تُحفظ الملفات الخام — `data/raw/`


In [ ]:
BASE_URL = "https://tenders.etimad.sa/Tender/AllSupplierTendersForVisitor"  # TODO: تأكدي الرابط + رقم الصفحة يضاف كيف (query param أو غيره)
PAGES_TO_SCRAPE = 20  # TODO: زيديها تدريجيًا بعد ما تتأكدين الكود شغال صحيح على عدد صفحات قليل
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

today_str = date.today().isoformat()


### 3. فتح المتصفح (headless) — تجاوز حماية Akamai

النقطة المهمة اللي جربناها وصارت تشتغل: نستخدم متصفح Chromium حقيقي (مو مجرد requests.get عادي)،
لأن Akamai يكتشف الطلبات اللي مالها بصمة متصفح حقيقي ويحظرها.

كمان: نستخدم `domcontentloaded` + `time.sleep(5)` بدل `networkidle` — لأن صفحات اعتماد فيها طلبات
تحديث مستمرة بالخلفية خلتها الـ networkidle ما توصل أبدًا لحالة "خلصت التحميل"، فتعلق الصفحة.


In [ ]:
def get_page(playwright):
    browser = playwright.chromium.launch(headless=True)
    context = browser.new_context(
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        ),
        locale="ar-SA",
    )
    page = context.new_page()
    return browser, page


def load_listing_page(page, url):
    page.goto(url, wait_until="domcontentloaded")
    time.sleep(5)  # نعطي الصفحة وقت تخلص تحميل البطاقات فعليًا (networkidle ما يشتغل هنا)
    return page


### 4. استخراج بطاقات المناقصات من الصفحة

**مهم جدًا — لازم تعدّلين الـ selectors:**
الأسماء أدناه (`.tender-card`, `.tender-title`...) هي أمثلة/placeholders بس.
افتحي الصفحة بمتصفحك، دوسي F12 (DevTools)، وشوفي الأسماء الحقيقية للعناصر (class/id) اللي فيها
اسم المناقصة، الجهة، تاريخ النشر، الرقم المرجعي، الموعد النهائي للاستفسار والتقديم، والقيمة.
غيّري القيم بالمتغيرات تحت (`CARD_SELECTOR`, `FIELD_SELECTORS`) بالأسماء الصحيحة.


In [ ]:
CARD_SELECTOR = ".tender-card"  # TODO: عدّلي حسب الفعلي بالصفحة

FIELD_SELECTORS = {
    "tender_name": ".tender-title",        # TODO
    "entity_full": ".tender-entity",       # TODO — النص الكامل زي "وزارة العدل - الديوان العام - إدارة المشتريات والعقود"
    "reference_number": ".tender-ref",     # TODO
    "publish_date": ".tender-publish-date",# TODO
    "inquiry_deadline_raw": ".tender-inquiry-deadline",  # TODO
    "submission_deadline": ".tender-submission-deadline",# TODO
    "tender_value": ".tender-value",       # TODO
    "category": ".tender-category",        # TODO
}


def extract_cards(page):
    cards = page.query_selector_all(CARD_SELECTOR)
    results = []
    for card in cards:
        record = {}
        for field_name, selector in FIELD_SELECTORS.items():
            el = card.query_selector(selector)
            record[field_name] = el.inner_text().strip() if el else None
        results.append(record)
    return results


### ملاحظة عن حقل `entity_full`

هذا الحقل يُحفظ **خام كما هو** بدون أي تعديل أو تصنيف — لأن Task 1 مسؤوليتها استخراج دقيق بس.
تصنيف الجهة الرئيسية كـ "Source" (لمتطلب الـ 10 مصادر) يصير بـ `02_profile_clean.ipynb` عن طريق
دكشنري (mapping) تراجعه أسيل بعد ما تشوف كل القيم الفريدة لـ `entity_full` بـ `value_counts()` —
مش بالتخمين وقت الاستخراج.


### 5. إصلاح اختلاف الهمزة بموعد الاستفسار

بعض بطاقات اعتماد تكتب "الاستفسار" بهمزة مختلفة عن بطاقات أخرى (مشكلة كتابة عربية شائعة:
"إستفسار" / "استفسار" / "الإستفسار"...) وهذا كان يخلي بعض الحقول تفوت وما تُستخرج.
هذا الـ regex يتقبّل كل الصيغ المحتملة.


In [ ]:
INQUIRY_DEADLINE_PATTERN = re.compile(
    r"(?:آخر|نهاية)\s+(?:موعد|تاريخ)\s+ل?(?:ا|إ)?ستفسار",
)


def normalize_inquiry_deadline(raw_text):
    \"\"\"يتأكد إن النص يطابق أحد أشكال كلمة الاستفسار المختلفة، ويرجع القيمة كما هي لو تطابقت،
    أو 'Unknown - extraction issue' لو ما قدر يتعرف على الحقل.\"\"\"
    if raw_text is None:
        return "Unknown - extraction issue"
    if INQUIRY_DEADLINE_PATTERN.search(raw_text) or raw_text.strip() != "":
        return raw_text.strip()
    return "Unknown - extraction issue"


### 6. منع التكرار (Duplicate cards)

اكتشفنا إن Akamai / تحديث الصفحة يخلي بعض البطاقات تتكرر بالـ DOM لأكثر من مرة.
الحل: نعتبر البطاقة "فريدة" بس لو عندها **تاريخ نشر + رقم مرجعي** الاثنين موجودين معًا —
هذا التركيب هو اللي يضمن إنها بطاقة حقيقية مختلفة، مو نسخة مكررة من نفس البطاقة.


In [ ]:
def deduplicate(records):
    seen = set()
    unique = []
    for r in records:
        key = (r.get("publish_date"), r.get("reference_number"))
        if key[0] and key[1] and key not in seen:
            seen.add(key)
            unique.append(r)
    return unique


### 7. تعبئة الحقول الفاضية بتسمية واضحة

بدل ما نسيب حقل فاضي (None) بلا تفسير، نعطيه تسمية توضح السبب — يفيدنا وقت التحقق بـ Task 3.


In [ ]:
def label_missing_fields(record):
    for key, value in record.items():
        if value is None or value == "":
            if key == "tender_value":
                record[key] = "N/A - Direct Purchase"
            else:
                record[key] = "Unknown - extraction issue"
    return record


### 8. تشغيل كل الخطوات على عدة صفحات (Pagination) وحفظ الملف الخام

هنا الجزء المهم لهدف "تنوّع المصادر": نتنقل بين عدة صفحات من القائمة العامة (بدون فلترة نشاط)
عشان نجمع مناقصات من أكبر عدد ممكن من الجهات والفئات المختلفة، مو بس أول صفحة.


In [ ]:
def build_page_url(base_url, page_number):
    """TODO: عدّلي هذي حسب طريقة الترقيم الفعلية بالموقع (مثلًا ?page=2 أو ?PageNumber=2).
    افتحي صفحة 2 بالمتصفح وشوفي شكل الرابط تحديدًا."""
    return f"{base_url}?page={page_number}"


def run_extraction(base_url, pages_to_scrape):
    all_raw_records = []
    with sync_playwright() as playwright:
        browser, page = get_page(playwright)
        for page_number in range(1, pages_to_scrape + 1):
            page_url = build_page_url(base_url, page_number)
            load_listing_page(page, page_url)
            page_records = extract_cards(page)
            print(f"صفحة {page_number}: استخرجنا {len(page_records)} بطاقة")
            all_raw_records.extend(page_records)
        browser.close()

    for r in all_raw_records:
        r["inquiry_deadline_raw"] = normalize_inquiry_deadline(r.get("inquiry_deadline_raw"))
        # ملاحظة: تصنيف entity_full لمصدر رئيسي (source_entity) يصير بـ Task 2، مو هنا

    unique_records = deduplicate(all_raw_records)
    final_records = [label_missing_fields(r) for r in unique_records]
    return final_records


records = run_extraction(BASE_URL, PAGES_TO_SCRAPE)
print(f"\nإجمالي البطاقات المستخرجة الفريدة عبر {PAGES_TO_SCRAPE} صفحة: {len(records)}")


### 9. حفظ الملف الخام في `data/raw/`

اسم الملف يحتوي المصدر والتاريخ، عشان كل يوم يكون له لقطة (snapshot) منفصلة.


In [ ]:
output_path = RAW_DIR / f"etimad_all_tenders_{today_str}.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"تم الحفظ في: {output_path}")


### 10. نظرة سريعة على تنوّع الجهات والفئات (أرقام تفيدكم بالعرض النهائي)

In [ ]:
distinct_entities_raw = sorted(set(r["entity_full"] for r in records))
distinct_categories_raw = sorted(set(r.get("category") for r in records if r.get("category")))

print(f"إجمالي المناقصات المستخرجة: {len(records)}")
print(f"عدد قيم entity_full الفريدة (خام، غير مصنّفة بعد): {len(distinct_entities_raw)}")
print(f"عدد الفئات/الأنشطة الفريدة: {len(distinct_categories_raw)}")
print()
print("عيّنة من الجهات:")
for e in distinct_entities_raw[:10]:
    print(" -", e)


> **ملاحظة:** هذا العدد أوّلي وغير مصنّف — بعد Task 2 (الدكشنري) بيصير عندكم عدد دقيق للجهات
> الرئيسية (Sources) بعد دمج الفروع تحت جهاتها الأصلية. الهدف مب بس الوصول لـ 10 — كل ما زاد
> عدد الصفحات المسحوبة (`PAGES_TO_SCRAPE`)، زاد التنوّع، وصار عندكم أرقام أقوى للعرض.


---
## قبل ما تشغّلين هذا فعليًا

1. سوّي: `pip install playwright` ثم `playwright install chromium` (مرة وحدة بس على جهازك)
2. افتحي tenders.etimad.sa بالمتصفح، دوسي F12، وحدّدي الـ selectors الحقيقية بخطوة 4
3. جربي أول مرة على صفحة واحدة بس (مو كل الفئات) للتأكد إن الاستخراج صحيح قبل توسعينه
4. لو واجهتي "N/A" أو "Unknown" بكثرة، رجعي لخطوة 4 و5 وتأكدي من الـ selectors/regex
